# A · Contratos

**📄 PÁGINA · Contratos** · https://rayalucaria.org/fdd_o26/python/stack/contratos/

**📓 NOTEBOOK · a_contratos.ipynb · celdas A.0–A.11**

La página trae la tabla de cerca, la figura y los resultados; aquí corres el código. Cada celda trae arriba qué compara y qué mirar, el código ya escrito, y abajo «Deberías ver» con la salida de referencia.

¿Te perdiste? Corre A.0 y salta a la celda que vamos.

- Corre en orden con Shift+Enter. Las celdas marcadas **💻 TERMINAL** no tienen código aquí: el comando va en la terminal de la ventana de `stack/`.
- Cada «Deberías ver» se corrió con `N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`. Tus números serán otros; lo que se mantiene es el patrón.
- Antes de entregar: *Clear All Outputs* y guarda.

## A.0 · Preparación

Corre esta celda primero, y otra vez cuando te pierdas: es **idempotente** (correrla dos veces deja lo mismo). Deja listo todo lo que usan las demás.

- **`N`**: el tamaño de los datos. Deja **una sola** línea `N = …` sin `#`. A.0 te recomienda uno según tu RAM, pero **no cambia `N` sola**.
- `version(libreria)` — la versión instalada. Deben ser `polars 2.0.0`, `duckdb 1.5.6`, `pandas 3.0.6`, `pydantic 2.14.0`.
- `sys.executable` — la ruta del Python que corre este kernel. Termina en `.venv/bin/python` o `python3`; si no, *Select Kernel* → *Jupyter Kernel…* → **fdd · stack (3.14)** (Haz 3b de la página).
- `datos.genera(N)` — escribe `datos/ventas_{N}.csv` y `.parquet` la primera vez; después los reusa. Con `N = 10_000_000` la primera vez tarda: ~400 MB en disco al generar; ~2.8 GB si además corres C.1 (bórralo con `rm datos/c1*`).
- `N_FILAS = min(N, 200_000)` — las carreras que crean **un objeto Python por fila** (A.8, A.9) usan a lo más 200 000 filas (A.8, la mitad): con más tardan minutos y el patrón no cambia.
- `csv.DictReader` — lee el CSV como una lista de diccionarios, uno por fila. **No convierte nada**: todo llega como texto (`str`), igual que de un formulario o una API. `islice(..., N_FILAS)` toma sólo las primeras `N_FILAS` filas, sin leer el resto.
- `FILA`, `MALA`, `VentaTD` y `VentaDC` — la fila de ejemplo, su versión con dos errores y dos contratos sin validación. Los usan A.4, A.5 y A.8; se definen aquí para que puedas saltar a cualquiera de esas celdas. A.4 y A.5 los explican.

In [ ]:
import csv
import sys
import tracemalloc
import warnings
from dataclasses import dataclass
from functools import partial
from importlib.metadata import version
from itertools import islice
from typing import NamedTuple, TypedDict

import polars as pl
from pydantic import BaseModel, TypeAdapter, ValidationError

import datos
from modelos import Venta

# Elige el tamaño según tu máquina: deja UNA línea sin #.
N = 1_000_000        # por defecto: cualquier laptop
# N = 100_000        # 8 GB de RAM o menos, o si una celda tarda más de 30 s
# N = 10_000_000     # 16 GB o más; hasta ~3 GB en disco en datos/

for libreria in ["polars", "duckdb", "pandas", "pydantic"]:
    print(libreria, version(libreria))
print(sys.executable)
datos.recomienda_n()
rutas = datos.genera(N)

# Las carreras de objetos Python (A.8, A.9) usan a lo más 200_000 filas.
N_FILAS = min(N, 200_000)
with open(rutas["csv"], encoding="utf-8") as f:
    filas = list(islice(csv.DictReader(f), N_FILAS))
print(f"N = {N:_} · N_FILAS = {N_FILAS:_} · primera fila: {filas[0]}")

# La fila de A.4–A.8, tal como llega de un CSV (todo texto), y sus dos contratos
# sin validación. Viven aquí para que «corre A.0 y salta» funcione.
FILA = {
    "fecha": "2025-01-03",
    "tienda": "Centro",
    "producto": "P001",
    "cantidad": "3",
    "precio": "12.50",
}


class VentaTD(TypedDict):
    fecha: str
    tienda: str
    producto: str
    cantidad: int
    precio: float


@dataclass
class VentaDC:
    fecha: str
    tienda: str
    producto: str
    cantidad: int
    precio: float


MALA = {**FILA, "precio": "abc", "cantidad": "-2"}

**Deberías ver:** (recortado: las rutas empiezan distinto en tu máquina)

```text
polars 2.0.0
duckdb 1.5.6
pandas 3.0.6
pydantic 2.14.0
…/09_python/stack/.venv/bin/python3
Tienes 31 GB de RAM → te recomendamos N = 1_000_000
Con más de 16 GB también puedes probar N = 10_000_000.
…/09_python/stack/datos/ventas_1000000.csv  1_000_000 filas  32.4 MB
…/09_python/stack/datos/ventas_1000000.parquet  1_000_000 filas  6.0 MB
N = 1_000_000 · N_FILAS = 200_000 · primera fila: {'fecha': '2025-11-07', 'tienda': 'Coyoacán', 'producto': 'P023', 'cantidad': '2', 'precio': '273.98'}
```

- Si dice `polars 1.…`, el kernel no es el del `.venv` de esta carpeta: corre `uv sync` en `stack/` y elige `.venv`.
- En `primera fila`, `cantidad` y `precio` van entre comillas: son **texto**. Toda esta página trata de quién los convierte, y cuándo se entera de que no se puede.

## A.1 · Una clase en un minuto

Pydantic y `dataclass` se escriben con `class`, y DataCamp aún no te enseñó clases. Aquí va lo justo para **leerlas**.

- `class Caja:` — define un **tipo** nuevo. Un **objeto** de ese tipo se crea llamándolo como función: `Caja(12.5)`.
- `def __init__(self, precio):` — la función que corre al crear el objeto. `self` es el objeto que se está creando.
- `self.precio = precio` — guarda un **atributo**: un nombre pegado al objeto, que lees con `caja.precio`.
- `class CajaConIva(Caja):` — **herencia**: `CajaConIva` tiene todo lo de `Caja` (aquí su `__init__`) sin escribirlo. `class Venta(BaseModel)` hereda así todo lo de Pydantic.
- `@dataclass` — una `@` sobre una definición la modifica: es un **decorador**. `@dataclass` lee las anotaciones `x: int` y te escribe el `__init__` y una forma legible de imprimir el objeto.

In [ ]:
class Caja:
    def __init__(self, precio):
        self.precio = precio


caja = Caja(12.5)
print(caja.precio)


class CajaConIva(Caja):  # hereda: tiene el __init__ de Caja sin escribirlo
    pass


print(CajaConIva(100).precio)


@dataclass
class Punto:
    x: int
    y: int


print(Punto(1, 2))

**Deberías ver:**

```text
12.5
100
Punto(x=1, y=2)
```

- `Punto(x=1, y=2)` lo escribió `@dataclass`. Con `print(caja)` verías `<__main__.Caja object at 0x…>`: una clase común no sabe imprimirse.
- `@dataclass` **no revisa** que `x` sea `int`: `Punto("a", None)` también funciona. Lo pruebas en A.4.

## A.2 · Python guarda la anotación y no la revisa

**Compara** la misma función con números y con textos. `total` vive en `modelos.py`:

```python
def total(precios: list[float]) -> float:
    return sum(precios)
```

- `precios: list[float]` y `-> float` son **anotaciones** (*type hints*): dicen qué tipo se espera y qué tipo regresa. `list[float]` es «lista de números con decimales».
- `total.__annotations__` — el diccionario donde Python guarda las anotaciones. Las guarda; **no las usa para revisar nada**.
- `traceback.extract_tb(error.__traceback__)[-1]` — el último paso antes del error: en qué función y en qué línea tronó.

**Qué mirar:** la llamada con textos entra sin queja. El error sale **adentro**, en la línea de `sum`, no en la llamada que estaba mal.

**La particularidad:** `sum` empieza en `0`, un `int`; por eso el mensaje dice `'int' and 'str'` y no menciona `float`.

In [ ]:
import traceback
from pathlib import Path

from modelos import total

print(total([12.5, 3]))
print(total.__annotations__)
try:
    total(["12.5", "3"])
except TypeError as error:
    ultimo = traceback.extract_tb(error.__traceback__)[-1]  # el último paso
    print(f"TypeError: {error}")
    archivo = Path(ultimo.filename).name
    print(f"  en {ultimo.name}(), línea {ultimo.lineno} de {archivo}: {ultimo.line}")

**Deberías ver:**

```text
15.5
{'precios': list[float], 'return': <class 'float'>}
TypeError: unsupported operand type(s) for +: 'int' and 'str'
  en total(), línea 27 de modelos.py: return sum(precios)
```

- `list[float]` está guardado, y aun así Python llamó a `total` con dos textos. El error apunta a la línea de `return sum(precios)`, dentro de `total`, lejos de quien pasó los textos.
- En código grande, la línea del error y la línea culpable pueden estar en archivos distintos. A.3 encuentra la culpable **sin correr nada**.

**📄 PÁGINA · Contratos · sección «Python guarda la anotación y no la revisa»**

## A.3 · mypy encuentra el mismo error sin correr

**💻 TERMINAL · en stack/ (comprueba con pwd)**

```bash
uv run mypy modelos.py
```

**Qué hace cada pieza:**

- `uv run` — corre el comando con el `.venv` de esta carpeta.
- `mypy` — un **revisor de tipos**: lee el archivo, compara cada llamada con las anotaciones y reporta lo que no cuadra. **No ejecuta ninguna línea.**
- `modelos.py` — al final trae `total(["12.5", "3"])`, bajo `if __name__ == "__main__":`.

Va en la terminal y no en una celda porque mypy lee **archivos**, no celdas.

**Deberías ver:**

```text
modelos.py:37: error: List item 0 has incompatible type "str"; expected "float"  [list-item]
modelos.py:37: error: List item 1 has incompatible type "str"; expected "float"  [list-item]
Found 2 errors in 1 file (checked 1 source file)
```

- `modelos.py:37` es la línea de la **llamada**, la culpable, no la de `sum`. Lo mismo que A.2, pero antes de correr y en el lugar correcto.
- Un error por cada texto de la lista: `List item 0` y `List item 1`.
- En VS Code, Pylance lo subraya en el editor si su revisión de tipos está en `basic` o más (`python.analysis.typeCheckingMode`; por defecto viene en `off`).

**📄 PÁGINA · Contratos · sección «Python guarda la anotación y no la revisa»**

## A.4 · La misma fila, tres contratos

**Compara** `TypedDict`, `dataclass` y Pydantic con la fila tal como llega de un CSV: `"precio": "12.50"`, texto. Los tres anotan `precio: float`.

`FILA`, `VentaTD` y `VentaDC` se definen en **A.0** (así puedes saltar aquí). Así se ven los dos contratos sin validación:

```python
class VentaTD(TypedDict):
    fecha: str
    tienda: str
    producto: str
    cantidad: int
    precio: float


@dataclass
class VentaDC:
    ...  # los mismos cinco campos
```

Y `Venta`, el de Pydantic, está en `modelos.py`: `class Venta(BaseModel)` con los mismos campos, `fecha: date`, y `Field(gt=0)` en `cantidad` y `precio`.

- `TypedDict` — un *type hint* para diccionarios: qué llaves lleva y de qué tipo. Lo usa mypy; **al correr es un `dict` común**. No compite con los *type hints*: es uno.
- `@dataclass` — una clase con `__init__` escrito por Python. Guarda lo que le des, tal cual.
- `BaseModel` de Pydantic v2 — una clase cuyo `__init__` **revisa y convierte** cada campo según su anotación. Esa revisión corre en Rust, en el paquete `pydantic-core`.
- `Venta.model_validate(FILA)` — crea una `Venta` desde un diccionario.
- `VentaTD(**FILA)` — `**` reparte el diccionario como argumentos con nombre: `precio="12.50"`, `cantidad="3"`…

**Qué mirar:** el tipo de `precio` que quedó guardado en cada uno.

In [ ]:
td = VentaTD(**FILA)
dc = VentaDC(**FILA)
pd_ = Venta.model_validate(FILA)
print("TypedDict ", repr(td["precio"]), type(td["precio"]).__name__)
print("dataclass ", repr(dc.precio), type(dc.precio).__name__)
print("Pydantic  ", repr(pd_.precio), type(pd_.precio).__name__)
print("Pydantic  ", repr(pd_.fecha))

**Deberías ver:**

```text
TypedDict  '12.50' str
dataclass  '12.50' str
Pydantic   12.5 float
Pydantic   datetime.date(2025, 1, 3)
```

- `TypedDict` y `dataclass` guardaron el **texto** `'12.50'`, aunque la anotación diga `float`. Sumarlo después truena como en A.2.
- Pydantic lo **convirtió**: `12.5` es `float`, y la fecha es un `date` de verdad, no un texto.

## A.5 · La fila mala: sólo uno se queja

**Compara** los tres con una fila con dos errores: `precio` es `"abc"` y `cantidad` es `"-2"`.

- `MALA = {**FILA, "precio": "abc", "cantidad": "-2"}` (definida en A.0) — copia `FILA` y reemplaza dos llaves.
- `ValidationError` — el error de Pydantic. Trae **todos** los errores de la fila, no sólo el primero, cada uno con su campo, su motivo y el valor que llegó.
- `Field(gt=0)` en `modelos.py` — una regla además del tipo: `gt` es *greater than*, estrictamente mayor que 0.

**Qué mirar:** el nombre entre corchetes de cada error, `type=…`. Es estable: lo puedes buscar en la documentación o filtrar en código.

In [ ]:
print("TypedDict  acepta:", VentaTD(**MALA)["precio"])
print("dataclass  acepta:", VentaDC(**MALA).precio)
try:
    Venta.model_validate(MALA)
except ValidationError as error:
    print(error)

**Deberías ver:** (recortado: cada error trae además una línea `For further information visit …`)

```text
TypedDict  acepta: abc
dataclass  acepta: abc
2 validation errors for Venta
cantidad
  Input should be greater than 0 [type=greater_than, input_value='-2', input_type=str]
precio
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='abc', input_type=str]
```

- `TypedDict` y `dataclass` aceptaron `"abc"` callados. El error saldrá después, lejos, cuando algo intente sumarlo.
- Pydantic reporta los dos juntos: `greater_than` en `cantidad` y `float_parsing` (no pudo leer un número) en `precio`.

## A.6 · Laxo o estricto

**Compara** el mismo dato, `{"cantidad": "3"}`, en los dos modos de Pydantic.

- **Laxo** (*lax*), el modo por defecto: convierte `"3"` → `3` si no pierde información. `"3.5"` → `int` sería error.
- **Estricto**, `strict=True`: el tipo debe llegar ya correcto; un texto donde va un `int` es error `int_type`.
- También se fija por campo (`Field(strict=True)`) o por modelo (`model_config = ConfigDict(strict=True)`).

**Qué mirar:** el mismo dato, dos respuestas. Laxo en la frontera, donde todo llega como texto (CSV, formularios); estricto entre partes de tu código, donde un texto en lugar de un número es un error tuyo.

In [ ]:
class Conteo(BaseModel):
    cantidad: int


print(Conteo.model_validate({"cantidad": "3"}))
try:
    Conteo.model_validate({"cantidad": "3"}, strict=True)
except ValidationError as error:
    print(error)

**Deberías ver:** (recortado: sin la línea `For further information visit …`)

```text
cantidad=3
1 validation error for Conteo
cantidad
  Input should be a valid integer [type=int_type, input_value='3', input_type=str]
```

- `cantidad=3` sin comillas: laxo lo convirtió. Estricto lo rechaza con `int_type` («debe ser entero»), no `int_parsing` («no pude leerlo»): ni siquiera intentó convertir.

## A.7 · Así se ve el código de Pydantic v1

Pydantic v2 salió el 2023-06-30. La IA leyó años de código v1 y lo sigue escribiendo. Corre, pero avisa:

| v1 (viejo) | v2 (hoy) |
|---|---|
| `@validator("precio")` | `@field_validator("precio")` |
| `.dict()` | `.model_dump()` |
| `.parse_obj(d)` | `.model_validate(d)` |
| `class Config:` | `model_config = ConfigDict(...)` |

- `warnings.catch_warnings(record=True)` — junta los avisos en una lista, `avisos`, en vez de mostrarlos sueltos.
- `warnings.simplefilter("always")` — muestra todos, aunque se repitan.
- `aviso.category.__name__` — el tipo del aviso; `str(aviso.message).split(".")[0]` — su primera frase.

**Qué mirar:** el nombre del aviso. Si lo ves en tu terminal, el código que te dieron es v1.

In [ ]:
from pydantic import validator

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")

    class VentaVieja(BaseModel):
        precio: float

        @validator("precio")
        def positivo(cls, v):
            assert v > 0
            return v

    print(VentaVieja(precio="12.5").dict())

for aviso in avisos:
    print(aviso.category.__name__, "·", str(aviso.message).split(".")[0])

**Deberías ver:**

```text
{'precio': 12.5}
PydanticDeprecatedSince20 · Pydantic V1 style `@validator` validators are deprecated
PydanticDeprecatedSince20 · The `dict` method is deprecated; use `model_dump` instead
```

- `{'precio': 12.5}`: el código v1 **funciona**. Por eso pasa desapercibido.
- `PydanticDeprecatedSince20`: obsoleto desde 2.0. El mensaje completo dice `to be removed in V3.0`: dejará de correr en Pydantic 3.

**📄 PÁGINA · Contratos · sección «Quién revisa, y cuándo»**

## A.8 · Carrera: cuánto cuesta una garantía

**Compara** seis formas de guardar las mismas filas: las buenas de la primera mitad de `filas` (`N_FILAS // 2`, a lo más 100 000; las que tienen `cantidad > 0`, ~98 %). La mitad, para que la celda no pase de ~10 s. Ejes: tiempo, memoria y si detecta `"abc"`.

- `NamedTuple` — una tupla con nombres: `v.precio` en vez de `v[4]`. No se puede modificar.
- `dataclass(slots=True)` — **slots**: guarda los atributos en lugares fijos dentro del objeto, sin un `dict` por objeto. Menos memoria; no puedes agregar atributos nuevos.
- `Venta.model_construct(**f)` — crea un modelo de Pydantic **sin validar**: confía en que los datos ya están bien.
- `tracemalloc` — cuenta la memoria que pide Python. `get_traced_memory()[0]` es la que sigue en uso al final.
- `crea_todas(crea)` — crea un objeto por cada fila buena. `partial(crea_todas, crea)` es una función nueva, sin argumentos, que llama `crea_todas(crea)`: lo que espera `cronometra`.
- `datos.cronometra(fn)` — corre `fn` tres veces y regresa el mejor tiempo. `datos.tabla(tiempos)` ordena de mejor a peor y agrega `veces_vs_mejor`.
- `replace_strict(memoria)` — busca cada `opcion` en el diccionario y pone su valor en una columna nueva.

**La trampa de medir:** `tracemalloc` hace todo **2–5× más lento**. Por eso hay dos pasadas: la 1 mide tiempo **sin** él; la 2, memoria **con** él.

**Qué mirar:** `veces_vs_mejor` y `detecta_abc`. Las cuatro primeras guardan texto; sólo `model_validate` convierte y revisa.

**La particularidad:** `model_construct` no valida y aun así es **más lento** que `model_validate`. `model_validate` corre en Rust (`pydantic-core`, compilado); `model_construct` está escrito en Python y arma el objeto campo por campo dentro del intérprete.

In [ ]:
class VentaNT(NamedTuple):
    fecha: str
    tienda: str
    producto: str
    cantidad: str
    precio: str


@dataclass(slots=True)
class VentaSlots:
    fecha: str
    tienda: str
    producto: str
    cantidad: str
    precio: str


# Sólo filas buenas: aquí se mide el costo de construir, no el de fallar.
mitad = filas[: N_FILAS // 2]  # la mitad: con todas, la celda pasa de 15 s
buenas = [f for f in mitad if f["cantidad"].isdigit() and int(f["cantidad"]) > 0]


def crea_todas(crea):
    return [crea(f) for f in buenas]


opciones = {
    "dict": lambda f: dict(f),
    "NamedTuple": lambda f: VentaNT(**f),
    "dataclass": lambda f: VentaDC(**f),
    "dataclass(slots=True)": lambda f: VentaSlots(**f),
    "Pydantic model_validate": Venta.model_validate,
    "Pydantic model_construct": lambda f: Venta.model_construct(**f),
}

# Pasada 1 · tiempo, SIN tracemalloc.
tiempos = {}
for nombre, crea in opciones.items():
    tiempos[nombre] = datos.cronometra(partial(crea_todas, crea))

# Pasada 2 · memoria, CON tracemalloc (hace todo más lento: por eso va aparte).
memoria, detecta = {}, {}
for nombre, crea in opciones.items():
    tracemalloc.start()
    objetos = crea_todas(crea)
    memoria[nombre] = round(tracemalloc.get_traced_memory()[0] / 1e6, 1)
    tracemalloc.stop()
    del objetos
    try:
        crea(MALA)
        detecta[nombre] = "no"
    except ValidationError:
        detecta[nombre] = "sí"

print(f"N_FILAS // 2 = {N_FILAS // 2:_} · filas buenas = {len(buenas):_}")
print(
    datos.tabla(tiempos).with_columns(
        pl.col("segundos").round(3),
        pl.col("opcion").replace_strict(memoria).alias("mb"),
        pl.col("opcion").replace_strict(detecta).alias("detecta_abc"),
    )
)

**Deberías ver:** (N_FILAS // 2 = 100_000; la celda tarda 9–12 s)

```text
N_FILAS // 2 = 100_000 · filas buenas = 97_886
shape: (6, 5)
┌──────────────────────────┬──────────┬────────────────┬───────┬─────────────┐
│ opcion                   ┆ segundos ┆ veces_vs_mejor ┆ mb    ┆ detecta_abc │
│ ---                      ┆ ---      ┆ ---            ┆ ---   ┆ ---         │
│ str                      ┆ f64      ┆ f64            ┆ f64   ┆ str         │
╞══════════════════════════╪══════════╪════════════════╪═══════╪═════════════╡
│ dict                     ┆ 0.045    ┆ 1.0            ┆ 18.8  ┆ no          │
│ dataclass(slots=True)    ┆ 0.131    ┆ 2.9            ┆ 7.8   ┆ no          │
│ dataclass                ┆ 0.157    ┆ 3.5            ┆ 11.8  ┆ no          │
│ NamedTuple               ┆ 0.189    ┆ 4.2            ┆ 10.2  ┆ no          │
│ Pydantic model_validate  ┆ 0.426    ┆ 9.5            ┆ 104.9 ┆ sí          │
│ Pydantic model_construct ┆ 0.568    ┆ 12.6           ┆ 99.5  ┆ no          │
└──────────────────────────┴──────────┴────────────────┴───────┴─────────────┘
```

- **Más garantías, más costo**: sólo `model_validate` detecta `"abc"`, y cuesta 5–10× lo de un `dict` en tiempo (según la corrida) y ~5.6× en memoria. La memoria sube porque crea objetos nuevos (`int`, `float`, `date`) en vez de reusar los textos.
- `slots=True` es la más chica de las que guardan texto: no lleva un `dict` por objeto. `slots=True`, `dataclass` y `NamedTuple` **casi empatan** en tiempo: entre corridas cambian de lugar.
- **Fuera del intérprete es rápido**: `model_construct` hace **menos** trabajo que `model_validate` y tarda **más** (1.3–1.8× en seis corridas), porque lo hace en Python.
- Tus números serán otros; lo que se mantiene es el patrón.

**📄 PÁGINA · Contratos · sección «Una garantía cuesta»**

## A.9 · Carrera: fila por fila o por columna

**Compara** tres formas de encontrar las filas malas en las mismas `filas`, ahora sucias: ~1 % con `cantidad` vacía, ~1 % con `cantidad ≤ 0`, y `"abc"` plantado en el precio de 1 de cada 1 000. Ejes: tiempo y **qué te reporta cada uno**.

- **Pydantic fila por fila** — un `try/except` por fila: guardas las buenas, anotas las malas y sigues.
- `TypeAdapter(list[Venta])` — valida un tipo que no es un modelo (aquí, una lista de ventas) en **una sola llamada**: el ciclo corre en Rust. Si una fila falla, **no regresa ninguna**: sólo un `ValidationError` con todos los motivos.
- **Polars por columna** — `cast(pl.Float64, strict=False)` convierte la columna entera; lo que no se puede leer queda `null` (vacío). `str.to_date(strict=False)` hace lo mismo con fechas. `is_null()` marca los nulos y `filter` se queda con las filas malas.
- `pl.any_horizontal(...)` — verdadero en una fila si **alguna** de las columnas lo es. `with_row_index("fila")` numera las filas desde 0.
- `filas_malas` — cuántas filas distintas reporta; `motivos` — cuántos «campo + motivo» trae. `motivos` puede ser mayor que `filas_malas`: una fila puede traer dos errores.

**El reloj** mide sólo validar: `texto` (las filas como tabla de Polars, todo en texto) se arma antes y no entra.

**La particularidad, Polars 2.0:** `cast(pl.Date)` desde texto **ya no existe**. La IA te lo escribirá; el error dice `It was removed in Polars 2.0. Use str.to_date() instead.`

In [ ]:
# Las mismas filas, con "abc" plantado en el precio de 1 de cada 1_000.
sucias = [dict(f) for f in filas]
for i in range(0, N_FILAS, 1_000):
    sucias[i]["precio"] = "abc"
texto = pl.DataFrame(sucias)  # todo texto, como llega; leer no entra al reloj


def por_fila():
    errores = []  # (fila, campo, motivo): uno por cada error de cada fila
    for i, f in enumerate(sucias):
        try:
            Venta.model_validate(f)
        except ValidationError as error:
            errores += [(i, e["loc"][0], e["type"]) for e in error.errors()]
    return errores


lista_de_ventas = TypeAdapter(list[Venta])


def de_golpe():
    try:
        lista_de_ventas.validate_python(sucias)
        return []
    except ValidationError as error:
        return error.errors()


def por_columna():
    return (
        texto.with_row_index("fila")
        .with_columns(
            pl.col("fecha").str.to_date(strict=False),
            pl.col("cantidad").cast(pl.Int64, strict=False),
            pl.col("precio").cast(pl.Float64, strict=False),
        )
        .filter(
            pl.any_horizontal(pl.col("fecha", "cantidad", "precio").is_null())
            | (pl.col("precio") <= 0)
            | (pl.col("cantidad") <= 0)
        )
    )


carrera = {
    "Pydantic fila por fila": por_fila,
    "TypeAdapter(list[Venta])": de_golpe,
    "Polars por columna": por_columna,
}
tiempos = {nombre: datos.cronometra(fn) for nombre, fn in carrera.items()}
errores_fila = por_fila()
errores_golpe = [(e["loc"][0], e["loc"][1], e["type"]) for e in de_golpe()]
malas_polars = por_columna()

print(f"N_FILAS = {N_FILAS:_}")
print(
    datos.tabla(tiempos).with_columns(
        pl.col("segundos").round(3),
        pl.col("opcion")
        .replace_strict(
            {
                "Pydantic fila por fila": len({i for i, _, _ in errores_fila}),
                "TypeAdapter(list[Venta])": len({i for i, _, _ in errores_golpe}),
                "Polars por columna": malas_polars.height,
            }
        )
        .alias("filas_malas"),
        pl.col("opcion")
        .replace_strict(
            {
                "Pydantic fila por fila": len(errores_fila),
                "TypeAdapter(list[Venta])": len(errores_golpe),
                "Polars por columna": 0,  # null no dice por qué
            }
        )
        .alias("motivos"),
    )
)
print("fila por fila:", errores_fila[:2])
print("de golpe     :", errores_golpe[:2])
print(malas_polars.head(3))

**Deberías ver:** (N_FILAS = 200_000; la celda tarda ~4 s)

```text
N_FILAS = 200_000
shape: (3, 5)
┌──────────────────────────┬──────────┬────────────────┬─────────────┬─────────┐
│ opcion                   ┆ segundos ┆ veces_vs_mejor ┆ filas_malas ┆ motivos │
│ ---                      ┆ ---      ┆ ---            ┆ ---         ┆ ---     │
│ str                      ┆ f64      ┆ f64            ┆ i64         ┆ i64     │
╞══════════════════════════╪══════════╪════════════════╪═════════════╪═════════╡
│ Polars por columna       ┆ 0.059    ┆ 1.0            ┆ 4299        ┆ 0       │
│ Pydantic fila por fila   ┆ 0.55     ┆ 9.4            ┆ 4299        ┆ 4299    │
│ TypeAdapter(list[Venta]) ┆ 0.558    ┆ 9.5            ┆ 4299        ┆ 4299    │
└──────────────────────────┴──────────┴────────────────┴─────────────┴─────────┘
fila por fila: [(0, 'precio', 'float_parsing'), (7, 'cantidad', 'int_parsing')]
de golpe     : [(0, 'precio', 'float_parsing'), (7, 'cantidad', 'int_parsing')]
shape: (3, 6)
┌──────┬────────────┬──────────┬──────────┬──────────┬────────┐
│ fila ┆ fecha      ┆ tienda   ┆ producto ┆ cantidad ┆ precio │
│ ---  ┆ ---        ┆ ---      ┆ ---      ┆ ---      ┆ ---    │
│ u32  ┆ date       ┆ str      ┆ str      ┆ i64      ┆ f64    │
╞══════╪════════════╪══════════╪══════════╪══════════╪════════╡
│ 0    ┆ 2025-11-07 ┆ Coyoacán ┆ P023     ┆ 2        ┆ null   │
│ 7    ┆ 2025-01-07 ┆ Oriente  ┆ P036     ┆ null     ┆ 480.67 │
│ 15   ┆ 2025-09-24 ┆ Coyoacán ┆ P033     ┆ null     ┆ 95.02  │
└──────┴────────────┴──────────┴──────────┴──────────┴────────┘
```

- Los tres encuentran **las mismas filas**. Lo que cambia es qué te dicen: Pydantic, la fila, el campo y el motivo (`float_parsing` en la fila 0, `int_parsing` en la 7); Polars, la fila con `null`, y **`"abc"` (fila 0) y vacío (fila 7) quedan iguales**: `motivos` es 0.
- `motivos` puede ser mayor que `filas_malas`: una fila puede traer dos errores (aquí ninguna trae dos, por eso empatan).
- `TypeAdapter` dice lo mismo que fila por fila, pero si una sola fila falla **no te regresa ninguna buena**: para quedarte con las buenas necesitas el ciclo o la columna.
- **Fila vs columna**: Polars trabaja la columna entera fuera del intérprete y gana por un orden de magnitud. Pydantic fila por fila y `TypeAdapter` **casi empatan**: los dos validan en Rust y el costo está en crear un objeto Python por fila.
- Tus números serán otros; lo que se mantiene es el patrón.

## A.10 · El contrato como JSON

- `Venta.model_json_schema()` — describe el modelo en **JSON Schema**: un formato estándar para decir qué campos, de qué tipo y con qué reglas. Lo leen otras máquinas: APIs (FastAPI lo publica solo), programas en otros lenguajes, y los LLM cuando les pides salida estructurada.
- `json.dumps(..., ensure_ascii=False)` — lo imprime como texto JSON.

**Qué mirar:** cómo se tradujo cada regla de `modelos.py`.

In [ ]:
import json

esquema = Venta.model_json_schema()
print(json.dumps(esquema["properties"]["precio"], ensure_ascii=False))
print(json.dumps(esquema["properties"]["fecha"], ensure_ascii=False))
print("required:", esquema["required"])

**Deberías ver:**

```text
{"exclusiveMinimum": 0, "title": "Precio", "type": "number"}
{"format": "date", "title": "Fecha", "type": "string"}
required: ['fecha', 'tienda', 'producto', 'cantidad', 'precio']
```

- `Field(gt=0)` se volvió `"exclusiveMinimum": 0`; `date`, `"format": "date"`. El contrato que escribiste en Python ya lo lee cualquier otro sistema.

**📄 PÁGINA · Contratos · sección «Fila por fila o por columna»**

## A.11 · ruff y pytest: el reemplazo de pycodestyle

**💻 TERMINAL · en stack/ (comprueba con pwd)**

```bash
uv run ruff check modelos.py
uv run pytest test_ventas.py
uv run ruff format --diff modelos.py
```

**Qué hace cada pieza:**

- `ruff check` — un **linter**: busca errores probables y estilo sin correr el código. Reemplaza a `pycodestyle` (el de DataCamp), `flake8` e `isort`. Sus reglas están en `pyproject.toml`, `[tool.ruff.lint]`.
- `pytest test_ventas.py` — corre cada función que empieza con `test_`. Un `assert` falso hace fallar la prueba; `pytest.raises(ValidationError)` exige que el error **sí** ocurra.
- `ruff format --diff` — muestra cómo reformatearía el archivo **sin cambiarlo**. Sin `--diff`, lo cambia.

**Deberías ver:** (recortado: ruff dibuja además el código alrededor de cada error)

```text
$ uv run ruff check modelos.py
F401 [*] `json` imported but unused
  --> modelos.py:9:8
…
E711 Comparison to `None` should be `cond is None`
  --> modelos.py:32:35
…
Found 2 errors.
[*] 1 fixable with the `--fix` option (1 hidden fix can be enabled with the `--unsafe-fixes` option).

$ uv run pytest test_ventas.py
…
collected 3 items

test_ventas.py ...                                                       [100%]

============================== 3 passed in 0.11s ===============================

$ uv run ruff format --diff modelos.py
1 file already formatted
```

- `F401`: `json` se importa y nadie lo usa. `E711`: `== None` donde va `is None`. `[*]` marca los que `ruff check --fix` arregla solo.
- `3 passed`: el contrato `Venta` convierte la fila buena y rechaza `"abc"` y `-5`.
- `1 file already formatted`: `ruff format` no tiene nada que cambiar. En código de IA, `--diff` te enseña los cambios con `-` y `+` antes de aceptarlos.

**📄 PÁGINA · Contratos · sección «Lo que DataCamp enseña y lo que se usa hoy»**

**📄 Regresa a la página Contratos.** Antes, *Restart* o *Shutdown* de este kernel: libera la memoria para el siguiente notebook.